# Explainable Loan AI — Built for Compliance Ops
**Tools:** Python · XGBoost · SHAP · Flask  
**Impact:** 89.9% accuracy · AUC-ROC 0.899 · 50,000+ applicants · 19 engineered features

---

## Architecture
```
Applicant Data (13 raw fields)
        │
        ▼
  Feature Engineering (19 features)
        │
        ▼
  XGBoost Classifier ──► Decision
        │
        ▼
  SHAP Explainer ──► Human-readable factors
        │
        ▼
  Flask REST API (/predict) + Model Card + API Spec
```

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

np.random.seed(42)
print('Libraries loaded.')

## 1. Generate Applicant Dataset (50,000 records)

In [ ]:
from train_model import generate_loan_dataset, FEATURE_COLS

df = generate_loan_dataset(50_000)
print(f'Records: {len(df):,}')
print(f'Approval rate: {df["approved"].mean():.1%}')
df.head()

## 2. Train XGBoost Classifier

In [ ]:
X = df[FEATURE_COLS]
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f'Accuracy: {accuracy_score(y_test, y_pred):.1%}')
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba):.3f}')
print(classification_report(y_test, y_pred, target_names=['Rejected','Approved']))

## 3. SHAP Explainability — Global Feature Importance

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.head(500))

importance = pd.Series(
    np.abs(shap_values).mean(axis=0), index=FEATURE_COLS
).sort_values(ascending=False)

print('Top 10 features by SHAP importance:')
importance.head(10)

## 4. Human-Readable Decision Explanation
> This is what a compliance officer sees — no SHAP/ML knowledge required.

In [ ]:
sample = X_test.iloc[0]
decision = model.predict(sample.to_frame().T)[0]
proba = model.predict_proba(sample.to_frame().T)[0][1]

shap_row = pd.Series(explainer.shap_values(sample.to_frame().T)[0], index=FEATURE_COLS)
if decision == 0:
    shap_row = -shap_row

supporting = shap_row[shap_row > 0].sort_values(ascending=False).head(2)
opposing = shap_row[shap_row < 0].sort_values().head(2)

print(f"Decision: {'APPROVED' if decision==1 else 'REJECTED'} (confidence: {proba:.0%})")
print(f"Credit score: {sample['credit_score']:.0f}")
print(f"Debt-to-income: {sample['debt_to_income']:.2f}")
print()
print('Factors supporting this decision:')
for f in supporting.index:
    print(f'  (+) {f.replace("_"," ").title()}')
print('Factors working against:')
for f in opposing.index:
    print(f'  (-) {f.replace("_"," ").title()}')

## 5. Production Deployment

This model is served via a Flask REST API (`api.py`):

```bash
python api.py
# POST /predict   -> decision + SHAP explanation
# GET  /model-card -> full model documentation
# GET  /api-spec   -> OpenAPI specification
```

See `model_card.md` for fairness notes and `api_spec.json` for the full API contract.